# Part 1 · Talking to a token generator

**Building Agentic AI — Day 1, 09:00–10:00**

By the end of this notebook you will have:

- made your first raw calls to the Gemini API,
- seen that the model produces **one token at a time** — nothing more magical than that,
- discovered that the model has **no memory** between calls (and how "chat" fakes it),
- watched it **make things up** with total confidence, and fixed that with context.

Everything an AI agent does this week is built on these four facts.

## 0. Setup

You should already have a virtual environment with the packages installed
(root README). Now get your **free Gemini API key** if you haven't:

1. Sign in at [aistudio.google.com](https://aistudio.google.com) with a
   **personal Google account**.
2. Click **Get API key** → **Create API key** (a new project is fine).
3. In the repo root, copy `.env.example` to `.env` and paste the key in:
   `GOOGLE_API_KEY=AIza…`

The free tier is all this week's Days 1–3 need — the notebooks cache and
retry around its rate limits. (On Thursday you'll switch to Google-provided
keys: same `.env` file, new value.) Keep the key secret — `.env` is
gitignored; leave it that way.

In [2]:
import os

from dotenv import load_dotenv

load_dotenv("../.env", override=True)

assert os.environ.get("GOOGLE_API_KEY"), (
    "No GOOGLE_API_KEY found. Copy .env.example to .env in the repo root "
    "and paste your key from https://aistudio.google.com"
)
print("API key loaded ✅")

API key loaded ✅


In [1]:
from google import genai
from google.genai import types

client = genai.Client()

MODEL = "gemini-3.5-flash-lite"

## 1. Your first API call

One HTTPS request. Text goes in, text comes out. This single function powers
everything we build this week.

In [3]:
response = client.models.generate_content(
    model=MODEL,
    contents="In one sentence, what is an AI agent?",
)
print(response.text)

An AI agent is an autonomous software system that perceives its environment, makes decisions, and takes actions to achieve specific goals without constant human intervention.


That's it. No magic. To be precise about what just happened:

1. Your prompt was cut into **tokens** (chunks of characters).
2. The model computed a probability for **every possible next token**.
3. One token was picked, appended, and step 2 ran again. And again. Until a stop token.

The model is a **next-token predictor**. It has no database of facts, no plan,
no goals. Keep this picture in your head all week — it explains almost every
weird behavior you'll ever see from an LLM.

## 2. Tokens, not words

The model doesn't see words or letters — it sees token IDs. Let's measure some text.

In [4]:
samples = [
    "Hello",
    "Hello, world!",
    "The game crashes after the update.",
    "Jocul se blochează după actualizare.",
    "supercalifragilisticexpialidocious",
    "🎮🔥🔥🔥",
]

for text in samples:
    n = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f"{n:>4} tokens · {text!r}")

   2 tokens · 'Hello'
   5 tokens · 'Hello, world!'
   8 tokens · 'The game crashes after the update.'
  12 tokens · 'Jocul se blochează după actualizare.'
  11 tokens · 'supercalifragilisticexpialidocious'
   4 tokens · '🎮🔥🔥🔥'


Things worth noticing:

- Tokens ≠ words. Common English words are often 1 token; rare words get chopped up.
- **Romanian costs more tokens** than English for the same meaning — tokenizers are
  trained mostly on English. Same idea, higher bill and higher latency.
- Emoji are surprisingly expensive.

Why care? Models have a **context window** (how many tokens fit in one call) and
you pay **per token**. When your agent on Friday reads 300 reviews, token math
becomes engineering, not trivia.

## 3. Watch it generate — literally token by token

The typing effect in chat apps is not an animation. It's the actual generation
loop, streamed to you.

In [5]:
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Pitch a brand-new indie video game in exactly 3 sentences.",
):
    print(chunk.text, end="", flush=True)

You play as a sentient glitch exploring a decaying digital museum where every exhibit is a forgotten video game from an alternate 1990s. By altering the source code of these dead worlds, you manipulate memories to prevent the system's impending total deletion. It's a melancholy, puzzle-platformer about nostalgia, digital archaeology, and the art of letting go.

## 4. Temperature: the chaos dial

At each step the model has a *probability distribution* over next tokens.
**Temperature** controls how that distribution is sampled:

- `0.0` → always take (nearly) the most likely token: focused, repeatable
- `~1.0` → sample proportionally: varied, creative
- `2.0` → flatten the distribution: chaos gremlin

Same prompt, three temperatures, three attempts each:

In [6]:
prompt = "Invent a name for a cozy game about gardening on the Moon. Reply with only the name."

for temp in [0.0, 1.0, 2.0]:
    names = []
    for _ in range(3):
        r = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=temp),
        )
        names.append(r.text.strip())
    print(f"temperature {temp}: {names}")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource has been exhausted (e.g. check quota).', 'status': 'RESOURCE_EXHAUSTED'}}

Rule of thumb for the rest of the week: **agents run cool** (low temperature).
When the model is deciding which tool to call, you want reliability, not poetry.

## 5. The model has no memory

Each API call is a blank slate. Watch it fail to remember:

In [ ]:
r1 = client.models.generate_content(
    model=MODEL,
    contents="Hi! My name is Ana and my favorite game is Dungeon Sprouts.",
)
print("Call 1:", r1.text)

In [ ]:
r2 = client.models.generate_content(
    model=MODEL,
    contents="What is my name, and which game do I like?",
)
print("Call 2:", r2.text)

Total amnesia — and yet every chat app appears to remember you. The trick:
**the entire conversation history is re-sent with every message.** The SDK's
`chats` helper does exactly that bookkeeping for you:

In [7]:
chat = client.chats.create(model=MODEL)

print(chat.send_message("Hi! My name is Ana and my favorite game is Dungeon Sprouts.").text)
print("---")
print(chat.send_message("What is my name, and which game do I like?").text)

Hi Ana! It's great to meet you. 

*Dungeon Sprouts* is such a fun game! I love the concept of little plant heroes exploring dungeons and fighting off pests. Do you have a favorite sprout character you like to play as, or a favorite level you've beaten recently?
---
Your name is Ana, and your favorite game is *Dungeon Sprouts*!


Proof there's no magic — the "memory" is just a growing list we resend every time:

In [8]:
for message in chat.get_history():
    text = message.parts[0].text.replace("\n", " ")
    print(f"{message.role:>6}: {text[:80]}")

  user: Hi! My name is Ana and my favorite game is Dungeon Sprouts.
 model: Hi Ana! It's great to meet you.   *Dungeon Sprouts* is such a fun game! I love t
  user: What is my name, and which game do I like?
 model: Your name is Ana, and your favorite game is *Dungeon Sprouts*!


📌 **This matters later:** on Day 3, agent "sessions" and "state" are exactly
this idea, made robust. Memory is not a property of the model — it's a property
of the *system you build around* the model.

## 6. When it doesn't know, it improvises

This week we work with a catalog of 20 indie games. Small detail: **we invented
all of them**. They do not exist. Let's ask the model about one anyway:

In [9]:
question = (
    "Who developed the video game 'Neon Drift Racers', and in what year "
    "was it released? Answer in one short sentence."
)
print(client.models.generate_content(model=MODEL, contents=question).text)

I am sorry, but there is no widely known or officially recognized video game titled *Neon Drift Racers* developed by a major studio, so its developer and release year cannot be definitively provided.


Run that cell a couple of times, and compare with your neighbors. You'll see one of two things:

- **A confident, specific, wrong answer** — a *hallucination*. The tokens are
  plausible, and plausible is all the model optimizes for.
- **A hedge** ("I'm not aware of this game...") — that's not knowledge either;
  it's post-training teaching the model when to fold.

Neither response involved *looking anything up*. So how do we get correct answers
about **our** games? We hand the model the facts:

In [ ]:
context = "Neon Drift Racers — arcade racing game, released 2024 by Velocity Forge, €19.99."

grounded = client.models.generate_content(
    model=MODEL,
    contents=(
        f"Answer using ONLY this context. If the answer is not in the context, say so.\n\n"
        f"Context: {context}\n\n"
        f"Question: {question}"
    ),
)
print(grounded.text)

You just did **grounding**: retrieval-augmented generation (RAG) in one cell.
This afternoon you'll build the *retrieval* part properly, and on Day 3 your
agent will do the whole loop by itself.

## 7. Exercises

Do them in order; ⭐⭐ ones are optional stretch goals.

**7.1 — Persona (⭐)** Use `system_instruction` (inside `GenerateContentConfig`)
to create a grumpy medieval blacksmith who reviews modern video games.
Ask for reviews of two games of your choice.

**7.2 — Temperature bingo (⭐)** Find a prompt where `temperature=0` gives the
*same* answer 5 times in a row, and `temperature=2` gives 5 *different* answers.

**7.3 — JSON by politeness (⭐⭐, remember this one!)** Ask the model to review an
imaginary game and reply *only* with JSON: `{"title": ..., "score": ..., "verdict": ...}`.
Run it 5 times and try `json.loads()` on each output. Count your successes.
Keep your score in mind — Part 4 fixes this properly.

**7.4 — Token economics (⭐)** Take any paragraph, translate it to Romanian
(the model can do it for you), and compute the EN/RO token ratio.

In [10]:
# 7.1 — your code here
prompt = "Create a Grumpy Medieval Blacksmith who reviews modern video games. Reply with only the review of Hearts of Iron IV and FIFA 22."
"""
for temp in [0.0, 1.0, 2.0]:
    names = []
    for _ in range(3):
        r = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=temp),
        )
        names.append(r.text.strip())
    print(f"temperature {temp}: {names}")
    """
print(client.models.generate_content(model=MODEL, contents=prompt).text)

**HEARTS OF IRON IV**

By the Hammer, what manner of sorcery is this? They call it a "grand strategy game," but it is naught but a cruel parchment nightmare. You sit in a damp chamber pushing little paper tokens of tanks and divisions across a map of the known world, staring at numbers until your eyes bleed into your porridge. I tried to command the realm of Germany—a proud, iron-lunged folk—and within three months my armies starved in the mud because some bureaucratic fool in Berlin forgot to forge enough locomotives. No hot iron. No sparks flying off the anvil. Just endless menus, sliders, and lads named "Churchill" shouting at me through a glowing glass box. **Two horseshoes out of five.** At least it gave me a headache fierce enough to make me forget my gout.

**FIFA 22**

Bah! They bring me this "FIFA" nonsense, claiming it is a contest of noble sport and athleticism. I looked upon the glowing screen, expecting men in chainmail to bludgeon one another over a pig's bladder until on

In [3]:
# 7.2 — your code here

for x in range(5):
    prompt = "What is the capital of France?"
    temp = 0.0
    names = []
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temp),
    )
    names.append(r.text.strip())
    print(f"{names}")

for x in range(5):
    prompt = "What characteristics did the first president of France have?"
    temp = 2.0
    names = []
    r = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temp),
    )
    names.append(r.text.strip())
    print(f"{names}")

NameError: name 'client' is not defined

In [6]:
# 7.3 — your code here
import json
ok = 0
for i in range(5):
    r = client.models.generate_content(
        model=MODEL,
        contents=(
            "Review the imaginary game 'Llama Mechanic 3000'. Reply ONLY with JSON: "
            '{"title": string, "score": number 1-10, "verdict": string}'
        ),
        config=types.GenerateContentConfig(temperature=1.5),  # high temp to stress it
    )
    try:
        json.loads(r.text)
        ok += 1
        print(f"run {i + 1}: ✅")
    except json.JSONDecodeError:
        print(f"run {i + 1}: 💥 → {r.text[:60]!r}…")
print(f"\n{ok}/5 parsed")

run 1: ✅
run 2: ✅
run 3: ✅
run 4: ✅
run 5: ✅

5/5 parsed


In [7]:
# 7.4 — your code here
paragraph = (
    "The new update completely changed the balance of the game. Weapons that "
    "used to be strong are now useless, and the community is not happy about it."
)

ro = client.models.generate_content(
    model=MODEL,
    contents=f"Translate to natural Romanian, reply with only the translation:\n{paragraph}",
).text.strip()

en_tokens = client.models.count_tokens(model=MODEL, contents=paragraph).total_tokens
ro_tokens = client.models.count_tokens(model=MODEL, contents=ro).total_tokens

print(ro, "\n")
print(f"EN: {en_tokens} tokens · RO: {ro_tokens} tokens · ratio {ro_tokens / en_tokens:.2f}×")

Noua actualizare a schimbat complet echilibrul jocului. Armele care înainte erau puternice sunt acum inutile, iar comunitatea nu este deloc mulțumită de acest lucru. 

EN: 31 tokens · RO: 41 tokens · ratio 1.32×


---
## ✅ Checkpoint

You can now explain: tokens, sampling & temperature, statelessness, and hallucination —
and you've already used the two fixes that define this course: **give the model context**
and **build structure around it**.

☕ **Take a break.** Next: *Part 2 — Meaning as geometry*, where words become vectors
and `numpy` becomes your friend.